Load the CMS catalog

In [0]:
import requests

catalog = requests.get("https://data.cms.gov/data.json", timeout=120).json()
print(catalog.keys())
print(len(catalog["dataset"]), "datasets in the catalog")

Find the Part D dataset

In [0]:
for section in ["dataset", "datasetSeries"]:
    for d in catalog.get(section, []):
        title = d.get("title", "")
        if "part d" in title.lower() and "prescriber" in title.lower():
            print(section, "|", repr(title))

In [0]:
partd_2021 = next(d for d in catalog["dataset"]
                  if d["title"] == "Medicare Part D Prescribers - by Provider : 2021-12-03")

for dist in partd_2021["distribution"]:
    print(dist.get("title"), "|", dist.get("mediaType"), "|", dist.get("downloadURL"))

Download all three years

In [0]:
import requests, os

prefix = "Medicare Part D Prescribers - by Provider : "

for year in [2019, 2020, 2021]:
    # find this year's entry (the dates in the titles vary, so match on the start + year)
    ds = next(d for d in catalog["dataset"]
              if d["title"].startswith(prefix) and str(year) in d["title"])

    # grab the CSV link
    url = next(dist["downloadURL"] for dist in ds["distribution"]
               if dist.get("mediaType") == "text/csv")

    dest = f"/Volumes/workspace/default/raw/partd_{year}.csv"
    if os.path.exists(dest):
        print(year, "already downloaded")
        continue

    print(year, "downloading", url)
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
    print(year, f"saved {os.path.getsize(dest) / 1e6:.0f} MB")

Keep only NC and save it as a table

In [0]:
from pyspark.sql import functions as F

for year in [2019, 2020, 2021]:
    df = spark.read.csv(f"/Volumes/workspace/default/raw/partd_{year}.csv",
                        header=True, inferSchema=True)
    nc = df.filter(F.col("Prscrbr_State_Abrvtn") == "NC").withColumn("year", F.lit(year))
    nc.write.mode("overwrite").saveAsTable(f"workspace.default.partd_nc_{year}")
    print(year, nc.count(), "NC prescribers")

In [0]:
df = spark.table("workspace.default.partd_nc_2021").toPandas()
df.head()

Load all three years into one table

In [0]:
df = (spark.table("workspace.default.partd_nc_2019")
      .unionByName(spark.table("workspace.default.partd_nc_2020"), allowMissingColumns=True)
      .unionByName(spark.table("workspace.default.partd_nc_2021"), allowMissingColumns=True)
      .toPandas())
print(df.shape)
df.head()
